In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

base = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/'

print(os.listdir(base))

In [ ]:
SKIP_TRAINING = True

In [ ]:
import pandas as pd

df = pd.read_csv(base + 'Data_Entry_2017.csv')

print("Shape:", df.shape)
df.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 14 disease names
DISEASE_COLS = [
    'Atelectasis', 'Cardiomegaly', 'Effusion',
    'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
    'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

# Count each disease
disease_counts = {}
for disease in DISEASE_COLS:
    count = df['Finding Labels'].str.contains(disease).sum()
    disease_counts[disease] = count

# No Finding count
no_finding = df['Finding Labels'].str.contains('No Finding').sum()

print("Disease Distribution:")
print("=" * 45)
for disease, count in sorted(disease_counts.items(), 
                              key=lambda x: x[1], reverse=True):
    pct = count/len(df)*100
    print(f"{disease:25} {count:6} ({pct:.1f}%)")
print(f"\n{'No Finding':25} {no_finding:6} ({no_finding/len(df)*100:.1f}%)")
print(f"{'Total':25} {len(df):6}")

In [ ]:
sorted_diseases = sorted(disease_counts.items(), 
                          key=lambda x: x[1], reverse=True)
diseases = [x[0] for x in sorted_diseases]
counts   = [x[1] for x in sorted_diseases]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
axes[0].bar(diseases, counts, color='steelblue', edgecolor='black')
axes[0].set_xticklabels(diseases, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Number of Images')
axes[0].set_title('Disease Distribution — NIH ChestX-ray14', 
                   fontweight='bold')
for i, (d, c) in enumerate(zip(diseases, counts)):
    axes[0].text(i, c+100, str(c), ha='center', fontsize=7)

# Pie chart — No Finding vs Disease
axes[1].pie(
    [no_finding, len(df)-no_finding],
    labels=['No Finding', 'Has Disease'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%', startangle=90)
axes[1].set_title('No Finding vs Disease', fontweight='bold')

plt.suptitle('NIH ChestX-ray14 — Class Imbalance Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
sorted_diseases = sorted(disease_counts.items(),
                          key=lambda x: x[1], reverse=True)
diseases = [x[0] for x in sorted_diseases]
counts   = [x[1] for x in sorted_diseases]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(diseases, counts, color='steelblue', edgecolor='black')
axes[0].set_xticklabels(diseases, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Number of Images')
axes[0].set_title('Disease Distribution — NIH ChestX-ray14',
                   fontweight='bold')
for i, (d, c) in enumerate(zip(diseases, counts)):
    axes[0].text(i, c+100, str(c), ha='center', fontsize=7)

axes[1].pie(
    [no_finding, len(df)-no_finding],
    labels=['No Finding', 'Has Disease'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%', startangle=90)
axes[1].set_title('No Finding vs Disease', fontweight='bold')

plt.suptitle('NIH ChestX-ray14 — Class Imbalance Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
DISEASE_COLS = [
    'Atelectasis', 'Cardiomegaly', 'Effusion',
    'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
    'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

def create_label_vector(finding_labels):
    label_vector = np.zeros(14, dtype=np.float32)
    for i, disease in enumerate(DISEASE_COLS):
        if disease in finding_labels:
            label_vector[i] = 1.0
    return label_vector

df['label_vector'] = df['Finding Labels'].apply(create_label_vector)

print("Verification:")
print(f"\nExample 1:")
print(f"Finding Labels: {df['Finding Labels'].iloc[0]}")
print(f"Label Vector:   {df['label_vector'].iloc[0]}")

print(f"\nExample 2 — Pneumonia patient:")
p = df[df['Finding Labels'].str.contains('Pneumonia')].iloc[0]
print(f"Finding Labels: {p['Finding Labels']}")
print(f"Label Vector:   {p['label_vector']}")
print(f"Pneumonia is index 6, value: {p['label_vector'][6]}")
print("\n✅ Label vectors ready!")

In [ ]:
from sklearn.model_selection import train_test_split

train_val_list = pd.read_csv(base + 'train_val_list.txt',
                              header=None, names=['Image Index'])
test_list      = pd.read_csv(base + 'test_list.txt',
                              header=None, names=['Image Index'])

train_val_df = df[df['Image Index'].isin(
    train_val_list['Image Index'])].reset_index(drop=True)
test_df = df[df['Image Index'].isin(
    test_list['Image Index'])].reset_index(drop=True)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("=" * 40)
print(f"Train: {len(train_df):,} images")
print(f"Val:   {len(val_df):,} images")
print(f"Test:  {len(test_df):,} images")
print("=" * 40)

In [ ]:
import cv2

def find_image(img_name, base_path):
    for folder in ['images_001', 'images_002', 'images_003',
                   'images_004', 'images_005', 'images_006',
                   'images_007', 'images_008', 'images_009',
                   'images_010', 'images_011', 'images_012']:
        path = base_path + folder + '/images/' + img_name
        if os.path.exists(path):
            return path
    return None

# Test
test_path = find_image(df['Image Index'].iloc[0], base)
print(f"Image found at: {test_path}")

# Sample X-Rays dikhao
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

sample_diseases = ['Pneumonia', 'Cardiomegaly', 'Effusion',
                   'Atelectasis', 'Nodule', 'Mass',
                   'Emphysema', 'Pneumothorax']

for i, disease in enumerate(sample_diseases):
    sample_row = df[df['Finding Labels'].str.contains(disease)].iloc[0]
    img_path   = find_image(sample_row['Image Index'], base)
    if img_path:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(disease, fontsize=11,
                          fontweight='bold', color='red')
    axes[i].axis('off')

plt.suptitle('NIH ChestX-ray14 — Sample X-Rays',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class NIHDataset(Dataset):
    def __init__(self, dataframe, base_path, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.base_path = base_path
        self.transform = transform
        self.folders   = [f'images_{str(i).zfill(3)}' 
                          for i in range(1, 13)]

    def __len__(self):
        return len(self.df)

    def find_image(self, img_name):
        for folder in self.folders:
            path = f"{self.base_path}{folder}/images/{img_name}"
            if os.path.exists(path):
                return path
        return None

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self.find_image(row['Image Index'])
        image    = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image    = cv2.resize(image, (224, 224))
        image    = np.stack([image, image, image], axis=-1)
        label    = row['label_vector'].astype(np.float32)
        if self.transform:
            image = self.transform(image=image)['image']
        return image, torch.tensor(label, dtype=torch.float32)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05,
                       scale_limit=0.05,
                       rotate_limit=10, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

train_dataset = NIHDataset(train_df, base, train_transform)
val_dataset   = NIHDataset(val_df,   base, val_transform)
train_loader  = DataLoader(train_dataset, batch_size=32,
                           shuffle=True,  num_workers=2)
val_loader    = DataLoader(val_dataset,   batch_size=32,
                           shuffle=False, num_workers=2)

print(f"✅ Train: {len(train_dataset):,} images")
print(f"✅ Val:   {len(val_dataset):,} images")
print(f"✅ Train batches: {len(train_loader)}")

img, lbl = train_dataset[0]
print(f"\nImage shape: {img.shape}")
print(f"Label shape: {lbl.shape}")
print(f"Label: {lbl}")

In [ ]:
import torchvision.models as tmodels
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

class NIHModel(nn.Module):
    def __init__(self, num_classes=14):
        super(NIHModel, self).__init__()
        self.densenet = tmodels.densenet121(pretrained=True)
        num_features  = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.densenet(x)

model = NIHModel(num_classes=14).to(device)
print(f"✅ Model ready on {device}!")
print(f"✅ Output: 14 disease probabilities")

In [ ]:
label_matrix = np.vstack(train_df['label_vector'].values)

pos_weights = []
print("Per-disease imbalance:")
print("=" * 55)
for i, disease in enumerate(DISEASE_COLS):
    pos    = label_matrix[:, i].sum()
    neg    = len(label_matrix) - pos
    weight = neg / pos if pos > 0 else 1.0
    pos_weights.append(weight)
    print(f"{disease:25} pos={int(pos):5} weight={weight:.1f}x")

pos_weight_tensor = torch.tensor(
    pos_weights, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5)

print("\n✅ Per-class weights applied!")
print("✅ Loss + Optimizer + Scheduler ready!")

In [ ]:
import time
from sklearn.metrics import roc_auc_score

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds  = []

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        all_labels.append(labels.cpu().detach().numpy())
        all_preds.append(
            torch.sigmoid(outputs).cpu().detach().numpy())

        if batch_idx % 200 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} "
                  f"Loss: {loss.item():.4f}")

    all_labels = np.vstack(all_labels)
    all_preds  = np.vstack(all_preds)

    # Mean AUC across all 14 diseases
    aucs = []
    for i in range(14):
        if len(np.unique(all_labels[:, i])) > 1:
            auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
            aucs.append(auc)

    return total_loss/len(loader), np.mean(aucs)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_labels = []
    all_preds  = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item()
            all_labels.append(labels.cpu().numpy())
            all_preds.append(
                torch.sigmoid(outputs).cpu().numpy())

    all_labels = np.vstack(all_labels)
    all_preds  = np.vstack(all_preds)

    # AUC per disease
    aucs = []
    print("\nAUC per disease:")
    for i, disease in enumerate(DISEASE_COLS):
        if len(np.unique(all_labels[:, i])) > 1:
            auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
            aucs.append(auc)
            print(f"  {disease:25} {auc:.4f}")

    mean_auc = np.mean(aucs)
    print(f"\n  {'Mean AUC':25} {mean_auc:.4f}")
    return total_loss/len(loader), mean_auc

print("Training functions ready!")

In [ ]:
if not SKIP_TRAINING:
    pass
"""
NUM_EPOCHS   = 5
best_val_auc = 0
save_path    = '/kaggle/working/nih_best_model.pth'

print("NIH Training Start!")
print(f"14 diseases | {len(train_dataset)} images")
print(f"Epochs: {NUM_EPOCHS}")
print("=" * 50)

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 30)

    train_loss, train_auc = train_one_epoch(
        model, train_loader, criterion, optimizer, device)

    print("\nValidating...")
    val_loss, val_auc = validate(
        model, val_loader, criterion, device)

    scheduler.step(val_loss)
    epoch_time = time.time() - start_time
    gap        = abs(train_auc - val_auc)

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train AUC: {train_auc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}   | Val AUC:   {val_auc:.4f}")
    print(f"  Overfit Gap: {gap:.4f} "
          f"{'✅ Good' if gap<0.05 else '⚠️ Watch'}")
    print(f"  Time: {epoch_time/60:.1f} minutes")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), save_path)
        print(f"  ✅ Best saved! AUC: {val_auc:.4f}")
    else:
        print(f"  No improvement. Best: {best_val_auc:.4f}")

print("\n" + "=" * 50)
print(f"Training Complete!")
print(f"Best Val AUC: {best_val_auc:.4f}")"""

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [28]:
import os

path = '/kaggle/working/nih_best_model.pth'

if os.path.exists(path):
    print("✅ Model exists")
    print("Size (MB):", os.path.getsize(path) / (1024*1024))
else:
    print("❌ Model NOT found")

✅ Model exists
Size (MB): 27.17342758178711
